# Topic 3 — Data Visualization
### Theory → tiny example → experiment.

**Why visualize before modeling?** Numbers in a table hide patterns your eyes catch instantly:
skewed distributions, outliers, class imbalance, correlated features. This is the last step of EDA
before you touch any ML algorithm.

Key ideas you're building intuition for:
- **Distribution** — how values of one variable are spread out
- **Outliers** — points far from the rest of the data
- **Correlation** — do two variables move together?
- **Variance** — how spread out a variable is
- **Trend** — a pattern over an ordered axis (e.g. time)
- **Class distribution** — how many samples belong to each label (critical for imbalanced data)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")   # nicer default look
rng = np.random.default_rng(42)

## 1. Line plot

Best for **trends over an ordered axis** — time, epochs during training, etc.

In [ ]:
epochs = np.arange(1, 21)
train_loss = 2.5 * np.exp(-epochs / 6) + rng.normal(0, 0.03, size=20)
val_loss = 2.5 * np.exp(-epochs / 7) + 0.15 + rng.normal(0, 0.04, size=20)

plt.figure(figsize=(6, 4))
plt.plot(epochs, train_loss, label="train_loss")
plt.plot(epochs, val_loss, label="val_loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Line plot: loss curves over training")
plt.legend()
plt.show()
# Look at where the two lines start diverging — that's the visual signature of overfitting.

## 2. Scatter plot

Best for showing the **relationship between two numeric variables** — and spotting outliers.

In [ ]:
hours_studied = rng.uniform(0, 10, 60)
score = 5 * hours_studied + rng.normal(0, 8, 60)
score = np.clip(score, 0, 100)

# Inject a couple of obvious outliers
hours_studied = np.append(hours_studied, [9, 0.5])
score = np.append(score, [10, 95])

plt.figure(figsize=(6, 4))
plt.scatter(hours_studied, score, alpha=0.7)
plt.xlabel("hours studied")
plt.ylabel("exam score")
plt.title("Scatter plot: relationship + outliers")
plt.show()
# The two points that don't follow the trend line are outliers — worth investigating, not always deleting.

## 3. Bar chart

Best for comparing **counts or averages across categories** — e.g. class distribution.

In [ ]:
df = pd.DataFrame({
    "label": rng.choice(["not_bullying", "bullying"], size=200, p=[0.75, 0.25]),
    "platform": rng.choice(["insta", "twitter", "youtube"], size=200),
})

plt.figure(figsize=(5, 4))
df["label"].value_counts().plot(kind="bar", color=["#4C72B0", "#DD8452"])
plt.title("Bar chart: class distribution")
plt.ylabel("count")
plt.xticks(rotation=0)
plt.show()
# This is exactly the plot you should make FIRST on your cyberbullying dataset —
# it tells you immediately whether you're dealing with class imbalance.

In [ ]:
# seaborn's countplot does the group-by + bar chart in one line
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="platform", hue="label")
plt.title("Bar chart: class distribution per platform")
plt.show()

## 4. Histogram

Best for the **distribution / shape** of a single numeric variable — is it normal? skewed? bimodal?

In [ ]:
text_lengths = rng.gamma(shape=2, scale=15, size=500)   # right-skewed, like real text-length data

plt.figure(figsize=(6, 4))
plt.hist(text_lengths, bins=30, color="#4C72B0", edgecolor="white")
plt.xlabel("text length (characters)")
plt.ylabel("frequency")
plt.title("Histogram: distribution of text length")
plt.show()
# Notice the long right tail — most posts are short, a few are very long. This is "skew".

## 5. Box plot

Best for comparing **spread, median, and outliers** across groups at a glance.
Box = 25th–75th percentile (IQR). Line in the box = median. Whiskers = typical range. Dots = outliers.

In [ ]:
box_df = pd.DataFrame({
    "text_length": np.concatenate([
        rng.gamma(2, 15, 150),                       # not_bullying
        rng.gamma(2, 22, 50),                         # bullying (tends longer, toy assumption)
    ]),
    "label": ["not_bullying"] * 150 + ["bullying"] * 50,
})

plt.figure(figsize=(5, 4))
sns.boxplot(data=box_df, x="label", y="text_length")
plt.title("Box plot: text length by class")
plt.show()
# The dots above the whiskers are outliers — unusually long posts for that class.

## 6. Distribution plots (KDE / displot)

A smoothed version of a histogram, plus easy comparison of multiple groups' distributions overlaid.

In [ ]:
plt.figure(figsize=(6, 4))
sns.kdeplot(data=box_df, x="text_length", hue="label", fill=True, alpha=0.4)
plt.title("KDE plot: text length distribution by class")
plt.show()
# Overlapping curves = the feature alone doesn't separate the classes well.
# Well-separated curves = that feature is likely useful for the model.

## 7. Heatmap & correlation

**Correlation** measures how strongly two numeric variables move together, from -1 to +1.
A heatmap visualizes a whole correlation matrix at once — useful for spotting redundant features.

In [ ]:
num_df = pd.DataFrame({
    "text_length": rng.gamma(2, 15, 200),
    "num_capital_letters": rng.poisson(5, 200),
    "num_exclamations": rng.poisson(1, 200),
})
# make num_capital_letters correlated with text_length on purpose, for illustration
num_df["num_capital_letters"] = (num_df["text_length"] * 0.3 + rng.normal(0, 2, 200)).clip(lower=0)

corr = num_df.corr()
print(corr)

plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Heatmap: feature correlation")
plt.show()
# Values near +1 -> strongly positively correlated (redundant features, watch out during modeling)
# Values near -1 -> strongly negatively correlated
# Values near 0  -> little to no linear relationship

## 8. Exercise

Put it all together on one small synthetic dataset.

In [ ]:
synthetic = pd.DataFrame({
    "label": rng.choice([0, 1], size=300, p=[0.8, 0.2]),
    "text_length": rng.gamma(2, 15, 300),
    "num_exclamations": rng.poisson(1, 300),
})

# --- Try it yourself ---
# 1. Bar chart: plot the class distribution of `label` (0 vs 1) and note the imbalance ratio.
# 2. Box plot: compare `text_length` across the two label groups.
# 3. Scatter plot: text_length (x) vs num_exclamations (y), colored by label
#    (hint: plt.scatter(..., c=synthetic["label"]))
# 4. Heatmap: compute synthetic[["label","text_length","num_exclamations"]].corr() and plot it.

---
### Next up: **Topic 4 — Mathematics for ML** (linear algebra, calculus, stats, probability — the concepts you'll actually use).

Say "next" when you're ready.